# Deep-OC-SORT Tracker


## Imports

In [1]:
from pathlib import Path

import cv2
import numpy as np
import supervision as sv

from deep_ocsort_vendor import DeepOCSortTracker
from gta_link_vendor import Tracklet
from orc_model.data import ClipDataset, PredictedClip


# --- configs

# data
CLIP_NAME = "IMG_2112"
TARGET_FPS = 30

## Helpers

In [2]:
# --- resample
def sample_frame_indices(native_fps: float, frame_count: int, target_fps: float) -> list[int]:
    """Evenly-spaced frame indices approximating `target_fps` given the video's native fps."""
    step = max(round(native_fps / target_fps), 1)
    return list(range(0, frame_count, step))

def read_frames(video_path: Path, frame_indices: list[int]) -> list[np.ndarray]:
    """Sequential decode + pick, rather than repeated `cap.set(...)` seeks."""
    wanted = set(frame_indices)
    cap = cv2.VideoCapture(str(video_path))
    frames_by_index = {}
    index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if index in wanted:
            frames_by_index[index] = frame
        index += 1
    cap.release()
    return [frames_by_index[i] for i in frame_indices if i in frames_by_index]

## Data

In [3]:
# --- get the data
dataset = ClipDataset.from_data_dir()
clip = dataset.get_clip(CLIP_NAME)

print(f"video: {clip.video_path}")
print(f"native fps: {clip.fps}, frame_count: {clip.frame_count}, resolution: {clip.resolution}")

# --- resample
frame_indices = sample_frame_indices(clip.fps, clip.frame_count, TARGET_FPS)
frames = read_frames(clip.video_path, frame_indices)

# --- cached detections
predicted_clip = PredictedClip.from_cache(clip.name)
raw_detections = predicted_clip.detections_by_frame

video: /Users/constantijn/Documents/eTHi.Link/dev/MVP-OKcamera/model/data/IMG_2112/video/IMG_2112.mp4
native fps: 30.0, frame_count: 1341, resolution: (1920, 1080)


## Detect + track

In [4]:
# detections
# like OC-SORT (and unlike ByteTrack), Deep-OC-SORT filters to a single
# confidence threshold internally (-> tracker's `det_thresh`), so raw
# detections are passed straight through below.
CONFIDENCE_THRESHOLD = 0.50     # -> tracker's `det_thresh`
IOU_THRESHOLD = 0.10            # kept positive -- OCM/OCR association gets unstable with negative IoU
EMBEDDING_OFF = False           # toggle off to ablate the appearance-embedding term -- note: downstream tracklet refinement (split/connect, see gta_link_vendor) needs real per-frame embeddings, so keep this False if you plan to build `tracklets` for refinement
CMC_OFF = False                 # toggle off to ablate camera-motion compensation
MASK_CROP = True                # toggle on to suppress background/other-instrument pixels in the embedder's crop, using each detection's segmentation mask instead of its raw bbox rectangle

# --- tracker
tracker = DeepOCSortTracker(
    det_thresh=CONFIDENCE_THRESHOLD,
    frame_rate=TARGET_FPS,
    max_age_seconds=20,             # mirrors the other notebooks' `lost_track_buffer=20*TARGET_FPS`
    min_hits=1*TARGET_FPS,        # mirrors the other notebooks' `minimum_consecutive_frames`
    iou_threshold=IOU_THRESHOLD,
    embedding_off=EMBEDDING_OFF,
    cmc_off=CMC_OFF,
    mask_crop=MASK_CROP,
)

# --- annots
mask_annotator = sv.MaskAnnotator(opacity=0.5, color_lookup=sv.ColorLookup.TRACK)
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 8916.39it/s]

In [5]:
annotated_frames = []           # frames with annots
track_ids_per_frame = []        # per-frame set of active track ids, for the summary below
track_positions_per_frame = []  # per-frame dict of tracker_id -> (x_center, y_center)
tracklets: dict[int, Tracklet] = {}  # per-track accumulation, for offline refinement (gta_link_vendor)

for frame_index, image in zip(frame_indices, frames):
    detections = raw_detections.get(frame_index, sv.Detections.empty())
    tracked = tracker.update(detections, image)

    if len(tracked) == 0:
        # no confirmed tracks this frame -- e.g. still within `min_hits` of
        # a fresh track, or an all-tracks-lost gap
        annotated_frames.append(image.copy())
        track_ids_per_frame.append(set())
        track_positions_per_frame.append({})
        continue

    # --- labels
    confidences = getattr(tracked, "confidence", None)
    if confidences is not None:
        labels = [
            f"#{tracker_id} {float(confidence):.2f}"
            for tracker_id, confidence in zip(tracked.tracker_id, confidences, strict=False)
        ]
    else:
        labels = [f"#{tracker_id}" for tracker_id in tracked.tracker_id]
    annotated = image.copy()
    annotated = mask_annotator.annotate(annotated, tracked)
    annotated = box_annotator.annotate(annotated, tracked)
    annotated = label_annotator.annotate(annotated, tracked, labels=labels)

    annotated_frames.append(annotated)
    track_ids_per_frame.append(set(tracked.tracker_id.tolist()))

    positions = {}
    for tracker_id, xyxy in zip(tracked.tracker_id, tracked.xyxy, strict=False):
        x_center = float(xyxy[0] + (xyxy[2] - xyxy[0]) / 2)
        y_center = float(xyxy[1] + (xyxy[3] - xyxy[1]) / 2)
        positions[int(tracker_id)] = (x_center, y_center)

    track_positions_per_frame.append(positions)

    # --- tracklets (raw per-frame appearance embeddings, for offline refinement)
    # `tracked.data["embedding"]` is only present when EMBEDDING_OFF=False
    # (see DeepOCSortTracker.update); when absent, still build times/scores/
    # bboxes below but leave `features` empty.
    embeddings = tracked.data.get("embedding")
    for row, tracker_id in enumerate(tracked.tracker_id):
        tracker_id = int(tracker_id)
        if tracker_id not in tracklets:
            tracklets[tracker_id] = Tracklet(track_id=tracker_id)
        tracklet = tracklets[tracker_id]
        tracklet.times.append(frame_index)
        tracklet.scores.append(float(tracked.confidence[row]) if confidences is not None else float("nan"))
        tracklet.bboxes.append(tracked.xyxy[row])
        if embeddings is not None:
            tracklet.features.append(embeddings[row])

print(f"tracked {len(annotated_frames)} frames")

tracked 1341 frames


## Track persistence

In [6]:
all_track_ids = sorted(set().union(*track_ids_per_frame)) if track_ids_per_frame else []
print(f"{len(all_track_ids)} unique track ids observed over {len(frames)} sampled frames\n")

# Show a compact timeline of tracks over time.
import plotly.graph_objects as go

frame_positions = np.array(frame_indices)

fig = go.Figure()
for track_id in all_track_ids:
    hits = [idx for idx, ids in enumerate(track_ids_per_frame) if track_id in ids]
    if hits:
        fig.add_trace(
            go.Scatter(
                x=[frame_positions[idx] for idx in hits],
                y=[track_id] * len(hits),
                mode="markers",
                marker=dict(size=8),
                name=f"track #{track_id}",
            )
        )

fig.update_layout(
    title="Track presence over sampled frames",
    xaxis_title="frame index",
    yaxis_title="track id",
    height=320,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

18 unique track ids observed over 1341 sampled frames



## Calibration -- appearance-embedding distance thresholds

Before any tracklet refinement (split/connect, `gta_link_vendor` -- a separate,
not-yet-built issue) we need to know whether the DINOv2-S embeddings computed
above can separate different surgical instruments at all, and if so, what
cosine-distance threshold to use. Upstream GTA-Link's defaults (`eps=0.7`,
`merge_dist_thres=0.4`) were tuned for OSNet person-ReID and carry no meaning
for this embedding space -- this section derives real numbers from this
project's data instead.

This is **offline-only calibration**: it produces the numbers that a future
refinement step -- and eventually an online track-linker, per
`docs/tracker-interface.md` -- will consume. See
`docs/plan-gta-link-tracklet-refinement.md` step 5.

Two distributions, both computed with no ground truth needed:
- **intra-tracklet**: pairwise distances between a tracklet's own features --
  "same instrument" by construction (a tracklet is one continuous track).
- **inter-tracklet**: mean distances between tracklets that overlap in time --
  "different instrument" by construction (the tracker never assigns two ids
  to the same detection in one frame).

Optional `MASK_CROP=True` vs `False` comparison (from the plan's step 5) is
skipped here: re-running the tracking loop with a different `MASK_CROP` means
re-running the DINOv2 embedder over the whole clip, which is exactly the
"expensive, skip unless cheap" case the issue calls out -- not attempted.

In [7]:
# --- intra-tracklet distances ("same instrument")
# Minimum tracklet length: >=2 frames, so there's at least one pairwise
# distance to compute per qualifying tracklet. Not filtering more
# aggressively (e.g. requiring long tracklets only) so the "same instrument"
# distribution isn't biased toward easy, long-lived tracks.
MIN_TRACKLET_LEN_FOR_CALIBRATION = 2

intra_distances = []
for tracklet in tracklets.values():
    if len(tracklet.features) < MIN_TRACKLET_LEN_FOR_CALIBRATION:
        continue
    feats = np.stack(tracklet.features).astype(np.float32)
    # features are already L2-normalized -> cosine distance = 1 - dot product
    dist = 1.0 - feats @ feats.T
    iu = np.triu_indices(len(feats), k=1)
    intra_distances.extend(dist[iu].tolist())

intra_distances = np.array(intra_distances)
n_qualifying = sum(1 for t in tracklets.values() if len(t.features) >= MIN_TRACKLET_LEN_FOR_CALIBRATION)
print(
    f"{len(intra_distances)} intra-tracklet pairwise distances from "
    f"{n_qualifying} qualifying tracklets (of {len(tracklets)} total, "
    f"min length {MIN_TRACKLET_LEN_FOR_CALIBRATION})"
)

2230543 intra-tracklet pairwise distances from 18 qualifying tracklets (of 18 total, min length 2)


In [8]:
# --- inter-tracklet distances ("different instrument", guaranteed correct)
# Two tracklets that share a frame index are necessarily two different
# instruments -- the tracker never assigns the same detection to two ids in
# one frame. This is exactly the overlap check `gta_link_vendor.refine.
# get_distance` uses to veto merges (forcing distance to 1.0); replicated
# here (not calling `get_distance` directly) because we want the *actual*
# mean pairwise distance for calibration, not the veto'd value.
def _tracklets_overlap(t1: Tracklet, t2: Tracklet) -> bool:
    return bool(set(t1.times) & set(t2.times))

inter_distances = []
tracklet_ids = list(tracklets.keys())
for i in range(len(tracklet_ids)):
    for j in range(i + 1, len(tracklet_ids)):
        t1, t2 = tracklets[tracklet_ids[i]], tracklets[tracklet_ids[j]]
        if not t1.features or not t2.features or not _tracklets_overlap(t1, t2):
            continue
        feats1 = np.stack(t1.features).astype(np.float32)
        feats2 = np.stack(t2.features).astype(np.float32)
        mean_dist = float((1.0 - feats1 @ feats2.T).mean())
        inter_distances.append(mean_dist)

inter_distances = np.array(inter_distances)
print(f"{len(inter_distances)} inter-tracklet mean distances from temporally-overlapping tracklet pairs")

94 inter-tracklet mean distances from temporally-overlapping tracklet pairs


In [9]:
# --- histogram: same-instrument vs. different-instrument distances
import plotly.graph_objects as go

# Pre-bin on the Python side rather than passing raw arrays to go.Histogram:
# plotly.js bins client-side at render time, so go.Histogram(x=...) embeds
# the full raw array (millions of floats for intra_distances) into the
# saved notebook JSON. Pre-binning with np.histogram keeps the saved output
# to ~40 numbers per trace instead.
bin_edges = np.histogram_bin_edges(np.concatenate([intra_distances, inter_distances]), bins=40)
intra_counts, intra_edges = np.histogram(intra_distances, bins=bin_edges)
inter_counts, inter_edges = np.histogram(inter_distances, bins=bin_edges)
intra_bin_centers = (intra_edges[:-1] + intra_edges[1:]) / 2
inter_bin_centers = (inter_edges[:-1] + inter_edges[1:]) / 2

fig = go.Figure()
fig.add_trace(go.Bar(x=intra_bin_centers, y=intra_counts, name="intra-tracklet (same instrument)", opacity=0.6))
fig.add_trace(go.Bar(x=inter_bin_centers, y=inter_counts, name="inter-tracklet (different instrument, overlapping)", opacity=0.6))
fig.update_layout(
    title="Cosine-distance distributions: same vs. different instrument",
    xaxis_title="cosine distance",
    yaxis_title="count",
    barmode="overlay",
    height=400,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

In [10]:
# --- heatmap: full pairwise tracklet distance matrix
from gta_link_vendor import get_distance_matrix

dist_matrix, matrix_track_ids = get_distance_matrix(tracklets)

# get_distance_matrix orders ids by `tracklets.keys()` (dict insertion order
# = first-seen order in the tracking loop above). Check whether that already
# coincides with ordering by first frame (`min(t.times)`) rather than assume
# it -- tracklet ids are only ever created the first time a track is
# confirmed, so in principle it should, but confirm rather than assume.
order_by_first_frame = sorted(matrix_track_ids, key=lambda tid: min(tracklets[tid].times))
if order_by_first_frame == matrix_track_ids:
    print("get_distance_matrix id order already matches first-frame order -- no reordering needed")
    ordered_ids = matrix_track_ids
    ordered_matrix = dist_matrix
else:
    print("get_distance_matrix id order did NOT match first-frame order -- reordering")
    ordered_ids = order_by_first_frame
    reindex = [matrix_track_ids.index(tid) for tid in ordered_ids]
    ordered_matrix = dist_matrix[np.ix_(reindex, reindex)]

fig = go.Figure(
    data=go.Heatmap(
        z=ordered_matrix,
        x=[str(tid) for tid in ordered_ids],
        y=[str(tid) for tid in ordered_ids],
        colorscale="Viridis",
        colorbar=dict(title="cosine distance"),
    )
)
fig.update_layout(
    title="Pairwise tracklet distance matrix (get_distance_matrix, ordered by first frame)",
    xaxis_title="track id",
    yaxis_title="track id",
    height=500,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

get_distance_matrix id order already matches first-frame order -- no reordering needed


In [11]:
# --- suggested merge_dist_thres + stop-gate check
intra_p95 = float(np.percentile(intra_distances, 95))
inter_p5 = float(np.percentile(inter_distances, 5))
suggested_merge_dist_thres = (intra_p95 + inter_p5) / 2

print(f"intra-tracklet p95 (same instrument, upper bound):    {intra_p95:.4f}")
print(f"inter-tracklet p5  (different instrument, lower bound): {inter_p5:.4f}")
print(f"suggested merge_dist_thres (midpoint):                 {suggested_merge_dist_thres:.4f}")

if intra_p95 >= inter_p5:
    print(
        "\nWARNING: intra-tracklet p95 >= inter-tracklet p5 -- the two "
        "distance distributions overlap heavily, i.e. there is no clean gap. "
        "DINOv2-S embeddings cannot cleanly separate surgical instruments in "
        "this data; any merge_dist_thres picked here would merge distinct "
        "instruments about as often as it reconnects fragments. Do NOT "
        "proceed to tracklet refinement with this embedder -- escalate to a "
        "bigger/more-robust backbone (dinov2_vitb14 or dinov2_vits14_reg) "
        "per docs/plan-gta-link-tracklet-refinement.md §4 before revisiting."
    )
else:
    print(f"\nClean gap of {inter_p5 - intra_p95:.4f} between distributions -- merging is viable.")

intra-tracklet p95 (same instrument, upper bound):    0.3359
inter-tracklet p5  (different instrument, lower bound): 0.3670
suggested merge_dist_thres (midpoint):                 0.3514

Clean gap of 0.0310 between distributions -- merging is viable.


**Reading the numbers above.** `intra p95` is a loose upper bound on how far
apart two crops of the *same* instrument can land; `inter p5` is a tight lower
bound on how close two crops of *different*, temporally co-visible instruments
can land -- the only negative signal here that's correct by construction, with
no ground truth needed. Their midpoint is the suggested `merge_dist_thres`. A
positive gap (`inter p5 > intra p95`) means DINOv2-S separates individual
instrument identities well enough that appearance-based reconnection is worth
trying; the wider the gap, the more headroom there is to pick a conservative
threshold (favoring leftover fragmentation over wrong merges, per the plan's
§4 risk that wrong merges are costlier than fragmentation). This number is a
*starting point* for `RefineConfig.merge_dist_thres` in the not-yet-built
refinement section (issue #13) -- it should still be sanity-checked against a
handful of eyeballed before/after merges once that section exists, not taken
as final.

On this run (`IMG_2112`, all 18 confirmed tracklets, `MASK_CROP=True`): intra
p95 = 0.3359, inter p5 = 0.3670, gap = 0.0310, suggested
`merge_dist_thres` &approx; 0.3514. The gap is positive but thin (~3% of the
0-2 cosine-distance range) -- DINOv2-S does separate instruments here, but not
by a wide margin, so refinement should stay conservative and lean toward
under-merging rather than trusting this threshold blindly. This is thinner
than issue #10's single-pair sanity check (0.977 vs 0.322 similarity on two
adjacent-frame crops), which is expected: that check compared one easy pair,
while this calibration pools every tracklet and every overlapping pair in the
clip, including harder within-tracklet variation (pose/lighting/occlusion
across a track's full lifetime) and harder cross-instrument pairs (visually
similar tool types seen together).

In [12]:
# outputs
OUTPUT_DIR = Path("../../output") / "deep_ocsort" / CLIP_NAME
OUTPUT_VIDEO_PATH = OUTPUT_DIR / f"tracked_{TARGET_FPS}fps{'_maskcrop' if MASK_CROP else ''}.mp4"
OUTPUT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

video_info = sv.VideoInfo(
    width=clip.resolution[0],
    height=clip.resolution[1],
    fps=TARGET_FPS,
    total_frames=len(annotated_frames),
)

with sv.VideoSink(str(OUTPUT_VIDEO_PATH), video_info) as sink:
    for annotated in annotated_frames:
        sink.write_frame(annotated)

print(f"wrote {OUTPUT_VIDEO_PATH}")

wrote ../../output/deep_ocsort/IMG_2112/tracked_30fps_maskcrop.mp4
